In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install transformers datasets evaluate accelerate

In [ ]:
# IMPORTS
import pandas as pd
import numpy as np
import evaluate

from datasets import Dataset
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
cols = [
    "id", "label", "statement", "subject", "speaker", "speaker_job",
    "state", "party", "barely_true_counts", "false_counts",
    "half_true_counts",
    "mostly_true_counts", "pants_fire_counts", "context"
]

train_df = pd.read_csv("train.tsv", sep="\t", header=None, names=cols)
valid_df = pd.read_csv("valid.tsv", sep="\t", header=None, names=cols)
test_df = pd.read_csv("test.tsv", sep="\t", header=None, names=cols)

In [ ]:
train_data = train_df[["statement", "label"]].copy()
valid_data = valid_df[["statement", "label"]].copy()
test_data = test_df[["statement", "label"]].copy()

In [ ]:
def convert_label(label):
    if label in ["true", "mostly-true"]:
        return "REAL"
    else:
        return "FAKE"

In [ ]:
for data in [train_data, valid_data, test_data]:
    data["target"] = data["label"].apply(convert_label)
    data["target_encoded"] = data["target"].map({
        "FAKE": 0,
        "REAL": 1
    })

print(train_data["target"].value_counts())

In [ ]:
train_hf = Dataset.from_pandas(train_data[["statement", "target_encoded"]])
valid_hf = Dataset.from_pandas(valid_data[["statement", "target_encoded"]])
test_hf = Dataset.from_pandas(test_data[["statement", "target_encoded"]])

In [ ]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels)["precision"],
        "recall": recall.compute(predictions=predictions, references=labels)["recall"],
        "f1": f1.compute(predictions=predictions, references=labels)["f1"],
    }

# DistilBert

In [ ]:
distilbert_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

distilbert_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
def distilbert_tokenize(batch):
    return distilbert_tokenizer(
        batch["statement"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

distilbert_train_hf = train_hf.map(distilbert_tokenize, batched=True)
distilbert_valid_hf = valid_hf.map(distilbert_tokenize, batched=True)
distilbert_test_hf = test_hf.map(distilbert_tokenize, batched=True)

In [ ]:
distilbert_train_hf = distilbert_train_hf.rename_column("target_encoded", "labels")
distilbert_valid_hf = distilbert_valid_hf.rename_column("target_encoded", "labels")
distilbert_test_hf = distilbert_test_hf.rename_column("target_encoded", "labels")

In [ ]:
distilbert_train_hf.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
distilbert_valid_hf.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
distilbert_test_hf.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
distilbert_training_args = TrainingArguments(
    output_dir="./distilbert_fake_news",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

In [ ]:
distilbert_trainer = Trainer(
    model=distilbert_model,
    args=distilbert_training_args,
    train_dataset=distilbert_train_hf,
    eval_dataset=distilbert_valid_hf,
    processing_class=distilbert_tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
!pip uninstall -y torchvision

In [ ]:
distilbert_trainer.train()

In [ ]:
# DISTILBERT VALIDATION RESULT

distilbert_results = distilbert_trainer.evaluate(distilbert_valid_hf)
print(distilbert_results)

# RoBERTa

In [ ]:
roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

roberta_model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

In [ ]:
def roberta_tokenize(batch):
    return roberta_tokenizer(
        batch["statement"],
        padding="max_length",
        truncation=True,
        max_length=256
    )
roberta_train_hf = train_hf.map(roberta_tokenize, batched=True)
roberta_valid_hf = valid_hf.map(roberta_tokenize, batched=True)
roberta_test_hf =  test_hf.map(roberta_tokenize, batched=True)

In [ ]:
roberta_train_hf = roberta_train_hf.rename_column("target_encoded", "labels")
roberta_valid_hf = roberta_valid_hf.rename_column("target_encoded", "labels")
roberta_test_hf = roberta_test_hf.rename_column("target_encoded", "labels")

In [ ]:
roberta_train_hf.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
roberta_valid_hf.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
roberta_test_hf.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
roberta_training_args = TrainingArguments(
    output_dir="./roberta_fake_news",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

In [ ]:
roberta_trainer = Trainer(
    model=roberta_model,
    args=roberta_training_args,
    train_dataset=roberta_train_hf,
    eval_dataset=roberta_valid_hf,
    processing_class=roberta_tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
roberta_trainer.train()

In [ ]:
roberta_results = roberta_trainer.evaluate(roberta_valid_hf)
print(roberta_results)

In [ ]:
roberta_test_results = roberta_trainer.evaluate(roberta_test_hf)
print(roberta_test_results)

In [ ]:
#  SAVE FINAL ROBERTA MODEL

roberta_trainer.save_model("./final_roberta_fake_news")
roberta_tokenizer.save_pretrained("./final_roberta_fake_news")

In [ ]:
!zip -r final_roberta_fake_news.zip final_roberta_fake_news

from google.colab import files
files.download("final_roberta_fake_news.zip")